# [16.3] Shapley Interactions with shapiq - Exercises

Implement exact pairwise Shapley interactions on complete coalition tables, then check parity against `shapiq`.

In [ ]:
import itertools
import math
import sys
from collections.abc import Callable, Mapping
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part3_shapley_interactions_shapiq"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_shapley_interactions_shapiq.tests as tests

Coalition = frozenset[int]

## Complete coalition tables

In [ ]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    if num_players <= 0:
        raise ValueError("num_players must be positive.")
    # YOUR CODE HERE
    raise NotImplementedError()


def coalition_values_from_function(
    num_players: int,
    value_fn: Callable[[Coalition], float],
) -> dict[Coalition, float]:
    # YOUR CODE HERE
    raise NotImplementedError()


def additive_game(weights: t.Tensor) -> dict[Coalition, float]:
    # YOUR CODE HERE
    raise NotImplementedError()


def interaction_game(
    num_players: int,
    *,
    pair: tuple[int, int] = (0, 1),
    pair_weight: float = 1.0,
    additive_weights: t.Tensor | None = None,
) -> dict[Coalition, float]:
    # YOUR CODE HERE
    raise NotImplementedError()

## Pairwise Shapley interactions

In [ ]:
def normalize_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> dict[Coalition, float]:
    values = {frozenset(key): float(value) for key, value in coalition_values.items()}
    expected = set(all_coalitions(num_players))
    missing = expected - set(values)
    if missing:
        raise ValueError(f"coalition value table is missing {len(missing)} coalitions.")
    return values


def pairwise_shapley_interactions(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    # YOUR CODE HERE
    raise NotImplementedError()


tests.test_additive_game_enumerates_complete_zero_interaction_table(additive_game, pairwise_shapley_interactions)
tests.test_interaction_game_recovers_target_pair_delta(interaction_game, pairwise_shapley_interactions)

## Target-pair report

In [ ]:
@dataclass(frozen=True)
class PairwiseInteractionReport:
    pair_interactions: t.Tensor
    target_pair: tuple[int, int]
    target_interaction: float
    max_spurious_interaction: float
    recovers_interaction: bool


def pairwise_interaction_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    target_pair: tuple[int, int] = (0, 1),
    expected_target: float = 1.0,
    tolerance: float = 1e-9,
) -> PairwiseInteractionReport:
    # YOUR CODE HERE
    raise NotImplementedError()


tests.test_pairwise_interaction_report_matches_reference_and_rejects_spurious_pairs(pairwise_interaction_report)

## shapiq parity

In [ ]:
@dataclass(frozen=True)
class ShapiqInteractionParityReport:
    exact_pair_interactions: t.Tensor
    shapiq_pair_interactions: t.Tensor
    max_abs_error: float
    matches_shapiq: bool
    shapiq_available: bool


def shapiq_pairwise_interactions(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    index: str = "SII",
) -> t.Tensor:
    # YOUR CODE HERE
    raise NotImplementedError()


def shapiq_interaction_parity_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    index: str = "SII",
    tolerance: float = 1e-6,
) -> ShapiqInteractionParityReport:
    # YOUR CODE HERE
    raise NotImplementedError()


tests.test_shapiq_interaction_parity_report_matches_exact_sii(shapiq_interaction_parity_report)

## Notebook contract

In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    # YOUR CODE HERE
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)

## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
